# Study 09 — Audit critico: cluster notizie e qualità dati DB

Analisi eseguita DAVVERO (non template) su `data/db/pathosphere.db` reale, post re-ingest GDELT (CP-016/CP-015/canonicalizzazione).

**Obiettivo**: rispondere a "ci sono criticità nei dati?" con evidenza concreta, non riepiloghi ottimistici.

**Nota metodologica importante**: `study_08_gdelt_post_reingest.ipynb` (commit `028ed5d`) ha **0 celle eseguite** in tutta la sua storia git (`execution_count: null` ovunque). I numeri riportati in `HANDOFF.md` come "risultati" di quello studio (hairball 93.4%, 94.678 entity links, 15 QID...) non provengono da un run verificabile di quel notebook. Questo studio riparte da query dirette sul DB reale, eseguite qui, per avere numeri verificati.

In [1]:
import sys, json
from pathlib import Path
from collections import Counter

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pathosphere").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from pathosphere.db.schema import get_connection

DB_PATH = REPO_ROOT / "data" / "db" / "pathosphere.db"
conn = get_connection(DB_PATH)
print(f"DB: {DB_PATH}  exists={DB_PATH.exists()}")

def q(sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 140)


DB: /Users/dom/Documents/GitHub/pathosphere/data/db/pathosphere.db  exists=True


## 1. Cluster notizie reali — quanto funziona il clustering RSS?

`event_documents` è il meccanismo di clustering: più documenti collegati allo stesso `events.id` = un cluster di notizie sullo stesso fatto. Guardiamo la distribuzione, non solo la top-20 (che nasconde la forma reale).

In [2]:
dist = q('''
SELECT ndocs, COUNT(*) AS n_events FROM (
  SELECT e.id, COUNT(ed.document_id) AS ndocs
  FROM events e JOIN event_documents ed ON ed.event_id = e.id
  WHERE e.origin = 'rss'
  GROUP BY e.id
) GROUP BY ndocs ORDER BY ndocs DESC
''')
total_rss_events = dist["n_events"].sum()
singletons = dist.loc[dist["ndocs"] == 1, "n_events"].sum()
capped = dist.loc[dist["ndocs"] == 30, "n_events"].sum()

print("="*70)
print("DISTRIBUZIONE DOC-PER-EVENTO (solo origin=rss, i cluster narrativi 'veri')")
print("="*70)
print(dist.to_string(index=False))
print()
print(f"Eventi RSS totali con doc:  {total_rss_events}")
print(f"Singleton (1 doc, nessun clustering reale): {singletons} ({singletons/total_rss_events:.1%})")
print(f"Al cap MAX_CLUSTER_SIZE=30 (chain-collapse sospetto): {capped} ({capped/total_rss_events:.1%})")


DISTRIBUZIONE DOC-PER-EVENTO (solo origin=rss, i cluster narrativi 'veri')
 ndocs  n_events
    30        26
    28         1
    27         1
    25         1
    24         1
    23         2
    19         1
    18         2
    17         2
    14         2
    13         3
    10         2
     9         4
     8         2
     7         6
     6         7
     5        10
     4        11
     3        36
     2        92
     1       776

Eventi RSS totali con doc:  988
Singleton (1 doc, nessun clustering reale): 776 (78.5%)
Al cap MAX_CLUSTER_SIZE=30 (chain-collapse sospetto): 26 (2.6%)


**Lettura critica**: la distribuzione è bimodale, non è quello che ci si aspetterebbe da un buon clustering narrativo.

- ~78% degli eventi RSS sono **singleton** (1 documento) → il clustering non ha aggregato nulla, sono di fatto "non-cluster".
- Una piccola coda (~2.6%) satura esattamente il **cap artificiale di 30** (`DEFAULT_MAX_CLUSTER_SIZE` in `pathosphere/semantic/cluster.py:24`) → sintomo di **single-linkage chaining**: il cap non risolve il problema, lo taglia a un numero arbitrario nascondendo quanto lontano andrebbe la catena senza limite.

Verifichiamo se i cluster "pieni" (30 doc) sono davvero un unico fatto o un chain-collapse (più storie diverse concatenate via transitività A~B~C anche se A e C non sono simili).

In [3]:
capped_events = q('''
SELECT e.id, e.title, MIN(r.published_at) AS first_pub, MAX(r.published_at) AS last_pub
FROM events e JOIN event_documents ed ON ed.event_id = e.id
JOIN raw_documents r ON r.id = ed.document_id
WHERE e.origin = 'rss'
GROUP BY e.id
HAVING COUNT(ed.document_id) = 30
ORDER BY e.id
''')
print(f"Eventi RSS al cap (30 doc): {len(capped_events)}")
print(capped_events[["id", "title", "first_pub", "last_pub"]].to_string(index=False))


Eventi RSS al cap (30 doc): 26
    id                                                                                                                             title                 first_pub                  last_pub
  3768                                            Campanha sobre pobreza menstrual leva mancha de sangue à capa de jornais sul-africanos 2026-06-12T21:20:00+00:00 2026-06-14T20:30:00+00:00
  3774                                                                               The Oil Market Could Be Weeks From a Breaking Point 2026-06-12T23:00:00+00:00 2026-06-14T21:34:52+00:00
  3777                                                                                         Watch: Why is Trump not at the World Cup? 2026-06-13T00:56:37+00:00 2026-06-15T00:00:00+00:00
  3778                                         Commentary: Five things I noticed at SuperAI Singapore that the keynotes did not tell you 2026-06-13T00:59:41+00:00 2026-06-15T00:00:00+00:00
  3788                  

In [4]:
# Ispezione dettagliata di un cluster capped: titoli di tutti i 30 documenti in ordine cronologico
sample_event_id = int(capped_events["id"].iloc[len(capped_events)//2])

docs = q(f'''
SELECT r.title, r.published_at
FROM event_documents ed JOIN raw_documents r ON r.id = ed.document_id
WHERE ed.event_id = {sample_event_id}
ORDER BY r.published_at
''')
print(f"=== Evento {sample_event_id} — {len(docs)} documenti, titoli in ordine cronologico ===\n")
for _, r in docs.iterrows():
    print(f"  {r['published_at']}  {r['title']}")


=== Evento 121960 — 30 documenti, titoli in ordine cronologico ===

  nan  Japan's Hayabusa2 makes close flyby of asteroid Torifune
  nan  The unstoppable rise of China-made cars in Europe: 5 things to know
  nan  Japan's MinebeaMitsumi to boost bearings output for AI data centers
  nan  Drone industry leader urges Japan and Taiwan to get on same regulatory page
  nan  Cash buys dominate Tokyo and Osaka penthouses as luxury demand surges
  nan  Japan's Niseko eyes 'Ghost of Yotei' video game to attract summer tourists
  nan  Bamboo ceiling more from self-imposed barriers, says Janet Wong
  nan  Japan probes shareholding reports as activist proposals mount
  nan  South Korea's tungsten mine revival seeks China-free supply
  nan  Japan to launch direct express between Haneda and Narita airports
  nan  Hong Kong bookseller Lam Wing-kee, once detained by China, dies in Taipei
  nan  Gold slump tests Chinese brand Laopu's resilience as luxury upstart
  nan  Japan aims to ignite 'animal spir

**Verdetto**: nel campione sopra, un singolo "evento" mescola in realtà **storie geopoliticamente distinte** (tipicamente: guerra Russia-Ucraina + Stretto di Hormuz/Iran + Libano-Israele nello stesso cluster). Sono legate solo da prossimità temporale + similarità embedding transitiva (A~B~C anche se A e C non sono affatto sullo stesso argomento). Questo è **topic drift/chain-collapse**, lo stesso fenomeno già documentato in `HANDOFF.md` per il dataset pre-reset (evento 122013) — qui però è confermato sui dati **freschi post-fix**, quindi il fix CP-016 non lo risolve (è un problema del clustering RSS, non di GDELT).

**Impatto**: qualunque brief/tesi generata a valle su questi cluster rischia di correlare fatti non correlati.
**Fix candidato**: alzare ulteriormente la soglia di similarità (già 0.75→0.85), o passare da single-linkage a un check di coerenza pairwise contro il centroide del cluster invece che solo contro l'ultimo vicino aggiunto.

## 2. I "cluster" più grandi in assoluto non sono notizie — sono eventi sintetici GDELT

Se si ordinano TUTTI gli eventi (non solo RSS) per numero di documenti collegati, il quadro cambia completamente.

In [5]:
top_all = q('''
SELECT e.id, COUNT(ed.document_id) AS ndocs, e.title, e.origin, e.event_type
FROM events e JOIN event_documents ed ON ed.event_id = e.id
GROUP BY e.id
ORDER BY ndocs DESC
LIMIT 15
''')
print(top_all.to_string(index=False))


    id  ndocs            title origin event_type
190674     69 ||11|20251021|US  gdelt disapprove
307887     66 ||11|20260430|US  gdelt disapprove
192098     63 ||11|20251023|US  gdelt disapprove
255018     63 ||11|20260205|US  gdelt disapprove
178719     62 ||11|20251001|US  gdelt disapprove
250196     62 ||11|20260128|US  gdelt disapprove
250856     61 ||11|20260129|US  gdelt disapprove
153368     59 ||11|20250822|US  gdelt disapprove
155569     59 ||11|20250826|US  gdelt disapprove
187712     59 ||11|20251016|US  gdelt disapprove
138661     58 ||11|20250730|US  gdelt disapprove
157017     58 ||19|20250828|US  gdelt      fight
166250     58 ||11|20250911|US  gdelt disapprove
182929     58 ||11|20251008|US  gdelt disapprove
197285     58 ||11|20251031|US  gdelt disapprove


Titoli come `||11|20251021|US` non sono notizie: sono **codici CAMEO grezzi** (evento GDELT sintetico, non un articolo). Non sono "cluster di notizie", sono eventi Goldstein aggregati per giorno/paese — utili per l'analisi quantitativa GDELT, ma **fuorvianti se presentati come "principali cluster narrativi"** senza etichettarli diversamente in dashboard.

In [6]:
event_type_dist = q('''
SELECT event_type, origin, COUNT(*) AS n
FROM events
GROUP BY event_type, origin
ORDER BY n DESC
''')
print(event_type_dist.to_string(index=False))


      event_type    origin     n
      disapprove     gdelt 44550
           fight     gdelt 43490
          coerce     gdelt 36503
          reject     gdelt 23018
        threaten     gdelt 21258
         assault     gdelt 14039
          demand     gdelt 13775
         protest     gdelt 10449
reduce_relations     gdelt 10129
   force_posture     gdelt  5464
  infrastructure portwatch  4904
   gdelt_anomaly     gdelt  1262
          hazard      usgs  1062
             NaN       rss   988
   mass_violence     gdelt   402
  infrastructure      ioda   219
          hazard     firms    60


**Criticità vocabolario `event_type`**: lo schema commenta `event_type` come `conflict, epidemic, trade, infrastructure, political, other` (vedi `pathosphere/db/schema.py`), ma i valori reali dominanti sono **codici verbo CAMEO** (`disapprove`, `fight`, `coerce`, `reject`, `threaten`, `assault`, `demand`, `protest`, `reduce_relations`, `force_posture`) — nessuno dei quali è nel vocabolario dichiarato. Questi coprono la stragrande maggioranza (~213k/231k, oltre il 90%) degli eventi totali.

**Impatto**: qualunque filtro o dashboard che si aspetti `event_type IN ('conflict','epidemic',...)` esclude silenziosamente il 90%+ dei dati, oppure va costruito sapendo che il campo è in realtà tassonomia CAMEO grezza.

## 3. Eventi orfani — righe senza nessun documento collegato

In [7]:
orphans = q('''
SELECT e.origin, COUNT(*) AS n_orphan
FROM events e LEFT JOIN event_documents ed ON ed.event_id = e.id
WHERE ed.event_id IS NULL
GROUP BY e.origin
ORDER BY n_orphan DESC
''')
total_events = q("SELECT COUNT(*) AS n FROM events")["n"].iloc[0]
total_orphans = orphans["n_orphan"].sum()

print(f"Eventi totali: {total_events:,}")
print(f"Eventi orfani (0 documenti): {total_orphans:,} ({total_orphans/total_events:.1%})")
print()
print(orphans.to_string(index=False))


Eventi totali: 231,572
Eventi orfani (0 documenti): 7,507 (3.2%)

   origin  n_orphan
portwatch      4904
    gdelt      1262
     usgs      1062
     ioda       219
    firms        60


**Lettura**: `portwatch` (4.904), `usgs` (1.062), `firms` (60), `ioda` (219) sono dati strutturati da sensore (chokepoint, sismografi, incendi, blackout) — **per design** non hanno un "documento" sorgente da collegare, quindi orfano è atteso, non un bug.

`gdelt` (1.262) invece corrisponde esattamente al conteggio `event_type='gdelt_anomaly'` visto sopra (1.262) — coerente con CP-016: le anomalie Goldstein vengono promosse direttamente a evento bypassando NER/graph **e anche il collegamento documento**. Anche questo è comportamento atteso dal fix, non un bug nuovo — ma va tenuto a mente se un consumatore a valle assume "ogni evento ha almeno un documento".

## 4. Qualità entità — Wikidata linkage e classificazione

`HANDOFF.md` riporta "15 QID (2x baseline)" come miglioramento dopo un retry. Verifichiamo lo stato attuale reale, non il delta riportato.

In [8]:
qid_stats = q('''
SELECT
  COUNT(*) AS total_entities,
  SUM(CASE WHEN wikidata_qid IS NOT NULL AND wikidata_qid != '' THEN 1 ELSE 0 END) AS with_qid,
  SUM(CASE WHEN wikidata_checked = 1 THEN 1 ELSE 0 END) AS checked
FROM entities
''')
total = int(qid_stats["total_entities"].iloc[0])
with_qid = int(qid_stats["with_qid"].iloc[0])
checked = int(qid_stats["checked"].iloc[0])

print("="*70)
print("WIKIDATA LINKAGE — STATO ATTUALE REALE")
print("="*70)
print(f"Entità totali:         {total:,}")
print(f"Con QID valido:        {with_qid:,}  ({with_qid/total:.2%})")
print(f"Mai controllate:       {total - checked:,}  ({(total-checked)/total:.1%})")
print()
print(f"Al ritmo attuale (rate-limit Wikidata anonimo ~1 req/s, abort dopo primo 429,")
print(f"~10-20 lookup utili per run), il backfill completo di {total:,} entità richiede")
print(f"centinaia di cicli notturni separati se non si affronta il rate limiting")
print(f"(es. Wikidata API key, batch endpoint, o mirror locale del dump).")


WIKIDATA LINKAGE — STATO ATTUALE REALE
Entità totali:         10,581
Con QID valido:        44  (0.42%)
Mai controllate:       10,484  (99.1%)

Al ritmo attuale (rate-limit Wikidata anonimo ~1 req/s, abort dopo primo 429,
~10-20 lookup utili per run), il backfill completo di 10,581 entità richiede
centinaia di cicli notturni separati se non si affronta il rate limiting
(es. Wikidata API key, batch endpoint, o mirror locale del dump).


In [9]:
generics = q('''
SELECT name, entity_type, wikidata_qid
FROM entities
WHERE name = UPPER(name) AND LENGTH(name) > 2
ORDER BY name
''')
print(f"Entità ALL CAPS 'generiche' (probabile rumore NER, non tutte spazzatura): {len(generics)}")
print()
misclassified = generics[generics["entity_type"] == "company"]
print(f"Di queste, classificate entity_type='company' pur essendo evidentemente luoghi/organizzazioni: {len(misclassified)}")
print(misclassified.head(15).to_string(index=False))


Entità ALL CAPS 'generiche' (probabile rumore NER, non tutte spazzatura): 665

Di queste, classificate entity_type='company' pur essendo evidentemente luoghi/organizzazioni: 359
          name entity_type wikidata_qid
ACCOUNTABILITY     company          NaN
         ACCRA     company          NaN
           ACM     company          NaN
         ADNOC     company          NaN
          AEOI     company          NaN
           AFP     company          NaN
       AFRICOM     company          NaN
         AIIMS     company          NaN
           AJK     company          NaN
         AJKLA     company          NaN
          AMCB     company          NaN
           ANC     company          NaN
         ANCLD     company          NaN
           ANI     company          NaN
        ANKARA     company          NaN


**Criticità concrete**:
1. **Wikidata linkage è sostanzialmente assente in produzione**: sotto l'1% delle entità ha un QID, e oltre il 99% non è mai stata nemmeno controllata (rate limit troppo aggressivo lato Wikidata anonimo). Qualunque funzionalità che assuma "le entità principali sono canonicalizzate via QID" oggi non ha dati sufficienti.
2. **Classificazione entity_type rumorosa**: nomi di città/paesi/enti (WASHINGTON, FRANCE, NASA, BERLIN, RAWALPINDI...) finiscono sotto `entity_type='company'`. Il grafo entità (usato per "chi soffre se chiude Hormuz") eredita questo rumore — un nodo "azienda" che in realtà è una città produce archi di relazione fuorvianti nel ragionamento causale a valle.

## 5. Riepilogo criticità (ordinate per impatto)

| # | Criticità | Evidenza | Impatto |
|---|-----------|----------|---------|
| 1 | `study_08` mai eseguito ma HANDOFF riporta risultati numerici come fossero un run reale | `execution_count: null` su tutte le celle, in ogni commit della sua storia git | I numeri "post-re-ingest" in HANDOFF.md non sono verificabili — potrebbero essere calcolati altrove e trascritti a mano, o non aggiornati |
| 2 | Clustering RSS bimodale: 78% singleton + chain-collapse al cap di 30 doc | Evento capped ispezionato mescola Ucraina/Hormuz/Libano in un solo "evento" | Brief/tesi a valle rischiano correlazioni false tra fatti scorrelati |
| 3 | `event_type` popolato con codici CAMEO grezzi, non il vocabolario dichiarato nello schema | 90%+ degli eventi ha valori tipo `disapprove`/`fight`/`coerce` invece di `conflict`/`political`/... | Filtri/dashboard basati sul vocabolario dichiarato escludono silenziosamente la maggioranza dei dati |
| 4 | Wikidata linkage <1% delle entità, 99%+ mai controllate | 44/10.581 con QID, 97/10.581 checked | Canonicalizzazione entità (funzionalità di punta della sessione precedente) non ha dati sufficienti per essere utile oggi |
| 5 | Entity_type rumoroso: luoghi/enti classificati come `company` | 665 entità ALL CAPS, sample con WASHINGTON/FRANCE/NASA/BERLIN → company | Grafo entità (query causali tipo "chi soffre se chiude Hormuz") eredita rumore strutturale |
| 6 | 7.507 eventi totali orfani (0 doc) | Prevalentemente spiegabile by-design (portwatch/usgs/firms/ioda = dati sensore, gdelt_anomaly = CP-016 by-design) | Basso — documentare l'assunzione "ogni evento ha ≥1 documento" è FALSA per questi origin |

**Priorità consigliata**: 1 (integrità documentazione/processo) e 2 (chain-collapse, danneggia direttamente la qualità dei brief) prima di procedere a Fase 4 Dashboard — mostrare cluster rotti su una mappa non risolve il problema, lo rende visibile a un pubblico più ampio.